# Test Fine-tuning + RAG sur TeleQnA
## Google Colab (GPU T4 gratuit)

**Prerequis:** Avoir déjà execute le notebook `fine_tuning_teleqna_colab.ipynb`
et avoir l'adapter LoRA dans Google Drive.

**1.** Runtime > Change runtime type > **T4 GPU**

**2.** Execute **Runtime > Run all**

**3.** Quand la cellule demande, uploade `telecom_test_split.json` et `corpus_rag.json`

In [ ]:
# ============================================================
# 1. Installer les dependances
# ============================================================
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft triton cut_cross_entropy unsloth_zoo
    !pip install trl==0.15.2 -q
    !pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
    !pip install --no-deps unsloth
    !pip install sentence-transformers

import torch
print(f"GPU: {torch.cuda.get_device_name()}" if torch.cuda.is_available() else "ERREUR: pas de GPU!")

In [ ]:
# ============================================================
# 2. Uploader les fichiers de donnees
# ============================================================
from google.colab import files
import json

print("Etape 1/2: Telechargez telecom_test_split.json")
uploaded = files.upload()
test_path = [f for f in uploaded.keys() if f.endswith('.json')][0]
with open(test_path, encoding="utf-8") as f:
    test_data = json.load(f)
print(f"Test: {len(test_data)} questions")

# Verifier le format
if isinstance(test_data, list) and len(test_data) > 0:
    keys = list(test_data[0].keys())
    print(f"Format OK: {keys}")
    assert "question" in keys, f"Cle 'question' manquante! Trousseau: {keys}"
    assert "reponse" in keys, f"Cle 'reponse' manquante!"
    assert "options" in keys, f"Cle 'options' manquante!"

print("\nEtape 2/2: Telechargez corpus_rag.json")
uploaded2 = files.upload()
corpus_path = [f for f in uploaded2.keys() if f.endswith('.json')][0]
with open(corpus_path, encoding="utf-8") as f:
    corpus_data = json.load(f)
print(f"Corpus: {len(corpus_data)} documents")
if isinstance(corpus_data, list) and len(corpus_data) > 0:
    print(f"Format OK: {list(corpus_data[0].keys())}")

In [ ]:
# ============================================================
# 3. Preparer les embeddings du corpus pour la recherche RAG
# ============================================================
from sentence_transformers import SentenceTransformer
import numpy as np

EMBED_MODEL = "paraphrase-multilingual-MiniLM-L12-v2"
embedder = SentenceTransformer(EMBED_MODEL)
print(f"Embeddings: {EMBED_MODEL}")

# Pre-calculer les embeddings du corpus
corpus_texts = [d.get("texte", "") for d in corpus_data]
print(f"Encodage de {len(corpus_texts)} documents du corpus...")
corpus_emb = embedder.encode(corpus_texts, show_progress_bar=True)
corpus_emb = corpus_emb / np.linalg.norm(corpus_emb, axis=1, keepdims=True)
print(f"Corpus encode: {corpus_emb.shape}")

In [ ]:
# ============================================================
# 4. Charger le modele fine-tune depuis Google Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

from unsloth import FastLanguageModel
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE_MODEL = "unsloth/phi-3-mini-4k-instruct-bnb-4bit"
ADAPTER_PATH = "/content/drive/MyDrive/telecom_lora_adapter"

print(f"Chargement du modele de base: {BASE_MODEL}")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

print(f"Chargement de l'adapter LoRA depuis: {ADAPTER_PATH}")
model = PeftModel.from_pretrained(model, ADAPTER_PATH)
FastLanguageModel.for_inference(model)
print("Modele pret!")

In [ ]:
# ============================================================
# 5. Fonction de recherche vectorielle
# ============================================================
def rechercher_documents(question, k=4):
    emb_q = embedder.encode([question])
    emb_q = emb_q / np.linalg.norm(emb_q)
    scores = np.dot(corpus_emb, emb_q.T).flatten()
    top_idx = np.argsort(scores)[-k:][::-1]
    docs = []
    for idx in top_idx:
        docs.append({
            "texte": corpus_texts[idx],
            "score": float(scores[idx]),
            "source": corpus_data[idx].get("source", ""),
        })
    return docs

# Test
sample = rechercher_documents("What is 5G NR?", k=2)
print(f"Recherche OK: {len(sample)} documents trouves")
for d in sample:
    print(f"  score={d['score']:.3f} | {d['texte'][:60]}...")

In [ ]:
# ============================================================
# 6. Evaluation FT+RAG sur le test set
# ============================================================
import time
import re

TOP_K = 4
nb_questions = min(50, len(test_data))  # 50 questions pour comparaison
test_subset = test_data[:nb_questions]
print(f"Evaluation FT+RAG sur {nb_questions} questions...")

ft_rag_results = []

for idx, item in enumerate(test_subset):
    question = item["question"]
    attendue = item["reponse"]
    options = item.get("options", [])

    # 1. Rechercher les documents pertinents
    docs = rechercher_documents(question, k=TOP_K)
    contexte = "\n\n".join([d["texte"] for d in docs])

    # 2. Generer la reponse avec contexte
    prompt = f"""### Contexte:
{contexte}

### Question:
{question}

### Reponse:"""

    start = time.time()
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=64,
        temperature=0.1,
        do_sample=True,
    )
    elapsed = time.time() - start
    reponse_predite = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

    # 3. Exact Match: le numero d'option correct apparait dans la prediction
    m = re.match(r"option\s+(\d+)", attendue.lower())
    opt_num = m.group(1) if m else None
    exact = opt_num is not None and opt_num in re.findall(r"\d+", reponse_predite)

    ft_rag_results.append({
        "question": question,
        "reponse_attendue": attendue,
        "reponse_predite": reponse_predite,
        "modele": "fine-tuned (Phi-3)",
        "approche": "Fine-tuning + RAG",
        "exact_match": exact,
        "temps": round(elapsed, 3),
    })

    if (idx + 1) % 10 == 0:
        print(f"  {idx+1}/{nb_questions} traitees")

# Metriques finales
n = len(ft_rag_results)
em = sum(1 for r in ft_rag_results if r["exact_match"])
temps_moy = sum(r["temps"] for r in ft_rag_results) / n
print(f"\nResultats FT+RAG sur {n} questions:")
print(f"  Exact Match: {em}/{n} ({em/n*100:.1f}%)")
print(f"  Temps moyen: {temps_moy:.3f}s")

In [ ]:
# ============================================================
# 7. Calculer le F1 semantique (comme dans le projet local)
# ============================================================
import numpy as np

def _semantic_similarity(text1, text2):
    emb = embedder.encode([text1, text2])
    emb = emb / np.linalg.norm(emb, axis=1, keepdims=True)
    return float(np.dot(emb[0], emb[1]))

def _clean_prediction(text):
    text = text.strip()
    patterns = [
        r"^[Bb]ased\s+on\s+(the\s+)?(context|document|specification)[^:]*:?\s*",
        r"^[Tt]he\s+(answer|correct|right)\s+(is|option|choice)[^:]*:?\s*",
        r"^[Aa]ccording\s+to\s+(the\s+)?(context|document)[^:]*:?\s*",
        r"^[Oo]ption\s+\d+\s*:\s*",
        r"^[Rr]eponse\s*:\s*",
        r"^[Aa]nswer\s*:\s*",
    ]
    for p in patterns:
        text = re.sub(p, "", text).strip()
    return text

# Build a quick lookup: question -> options
test_lookup = {q["question"]: q.get("options", []) for q in test_data}

for r in ft_rag_results:
    attendue = r["reponse_attendue"]
    options = test_lookup.get(r["question"], [])

    # Get correct option text
    m = re.match(r"option\s+(\d+)", attendue.lower())
    opt_num = m.group(1) if m else None
    correct_text = ""
    if opt_num and options:
        idx = int(opt_num) - 1
        if 0 <= idx < len(options):
            correct_text = options[idx].strip()

    cleaned = _clean_prediction(r["reponse_predite"]) or r["reponse_predite"]
    compare = correct_text or (attendue.split(":", 1)[-1].strip() if ":" in attendue else attendue)
    f1_val = _semantic_similarity(cleaned, compare) if compare else 0.0

    threshold = 0.4
    r["f1_score"] = round(f1_val, 4)
    r["precision"] = round(1.0 if f1_val >= threshold else 0.0, 4)
    r["recall"] = round(1.0 if f1_val >= threshold else 0.0, 4)

f1_moy = sum(r["f1_score"] for r in ft_rag_results) / n
print(f"  F1 moyen: {f1_moy:.4f}")
print(f"  Precision: {sum(r['precision'] for r in ft_rag_results) / n:.4f}")
print(f"  Recall: {sum(r['recall'] for r in ft_rag_results) / n:.4f}")

In [ ]:
# ============================================================
# 8. Sauvegarder sur Google Drive
# ============================================================
import json

resume = [{
    "modele_approche": "fine-tuned (Phi-3) / Fine-tuning + RAG",
    "modele": "fine-tuned (Phi-3)",
    "approche": "Fine-tuning + RAG",
    "exact_match_rate": round(em / n, 4),
    "f1_moyen": round(f1_moy, 4),
    "precision": round(sum(r["precision"] for r in ft_rag_results) / n, 4),
    "recall": round(sum(r["recall"] for r in ft_rag_results) / n, 4),
    "temps_moyen": round(temps_moy, 3),
    "nb_questions": n,
}]

save = {
    "resume": resume,
    "details": {"fine-tuned (Phi-3) / Fine-tuning + RAG": ft_rag_results},
}

save_path = "/content/drive/MyDrive/telecom_lora_adapter"
with open(f"{save_path}/resultats_finetuning_rag.json", "w", encoding="utf-8") as f:
    json.dump(save, f, indent=2, ensure_ascii=False)
print(f"Resultats sauvegardes dans: {save_path}/resultats_finetuning_rag.json")
print(f"\nResume:")
for r in resume:
    print(f"  {r['modele_approche']}")
    print(f"  EM={r['exact_match_rate']:.1%} | F1={r['f1_moyen']:.4f} | Temps={r['temps_moyen']:.3f}s | n={r['nb_questions']}")

## Apres l'execution

1. Les resultats sont dans **Google Drive** > `telecom_lora_adapter/resultats_finetuning_rag.json`
2. Copie ce fichier dans `project-rag/reports/`
3. Execute `python scripts/merger_ft_rag.py` pour fusionner avec les resultats existants
4. Execute `python scripts/generer_graphiques.py` pour regenerer les graphiques